# Split dataset - train / val

Divise les images annotées (export Label Studio YOLO) en train (80 %) et val (20 %),
puis les copie dans `data/processed/` avec la structure attendue par Ultralytics.

In [ ]:
import random
import shutil
from pathlib import Path

SEED = 42
VAL_RATIO = 0.2
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}

PROJECT_ROOT = Path.cwd().parent
SRC_IMAGES = PROJECT_ROOT / "data" / "labels" / "images"
SRC_LABELS = PROJECT_ROOT / "data" / "labels" / "labels"
PROCESSED = PROJECT_ROOT / "data" / "processed"

# Collect stems that have both image and label
stems = sorted([
    p.stem for p in SRC_IMAGES.iterdir()
    if p.suffix in IMAGE_EXTENSIONS and (SRC_LABELS / (p.stem + ".txt")).exists()
])

print(f"Paires image+label trouvées : {len(stems)}")

In [2]:
random.seed(SEED)
random.shuffle(stems)

n_val = int(len(stems) * VAL_RATIO)
val_set = set(stems[:n_val])
trn_set = set(stems[n_val:])

print(f"Train : {len(trn_set)}  |  Val : {len(val_set)}")

Train : 180  |  Val : 45


In [ ]:
def find_image(src_dir, stem):
    for ext in IMAGE_EXTENSIONS:
        p = src_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def copy_split(stems_subset, split_name):
    img_dst = PROCESSED / split_name / "images"
    lbl_dst = PROCESSED / split_name / "labels"
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    for stem in stems_subset:
        src_img = find_image(SRC_IMAGES, stem)
        shutil.copy(src_img, img_dst / src_img.name)
        shutil.copy(SRC_LABELS / f"{stem}.txt", lbl_dst / f"{stem}.txt")

    print(f"  {split_name}: {len(stems_subset)} fichiers copiés")


# Réinitialise processed/ pour un split propre
if PROCESSED.exists():
    shutil.rmtree(PROCESSED)

copy_split(trn_set, "train")
copy_split(val_set, "val")
print("Terminé.")